# Query Pipeline — COSORA

Busqueda hibrida (Dense + BM25 con RRF), prompt builder y generacion con LLM.

**Prerequisito:** ejecutar `ingestion_pipeline.ipynb` primero.

## 0. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q chromadb sentence-transformers rank_bm25 transformers accelerate python-dotenv


In [4]:

import os
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import torch
from transformers import pipeline
from dotenv import load_dotenv

# Cargar variables de entorno desde Drive
env_path = '/content/drive/MyDrive/variablentorno/.env'
if os.path.exists(env_path):
    load_dotenv(env_path)
    print('✅ Variables de entorno cargadas')

✅ Variables de entorno cargadas


In [5]:
# Config
CHROMA_PATH     = '/content/drive/MyDrive/RAG_UPC_Final_project/chroma_db'
COLLECTION_NAME = 'cosora_chunks'
EMBED_MODEL     = 'intfloat/multilingual-e5-base'

# Backend de generacion: "local" o "openai"
GEN_BACKEND = 'openai'
LOCAL_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

# Conectar
client_chroma = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = client_chroma.get_collection(COLLECTION_NAME)
embed_model   = SentenceTransformer(EMBED_MODEL)

print(f'ChromaDB: {collection.count()} chunks')
print(f'Backend: {GEN_BACKEND}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB: 268 chunks
Backend: openai


## 1. Retrieval hibrido

In [6]:
def dense_search(query, k=20):
    q_vec = embed_model.encode(f"query: {query}").tolist()
    res   = collection.query(query_embeddings=[q_vec], n_results=k)
    return [
        {'text': doc, 'meta': meta, 'rank_dense': i}
        for i, (doc, meta) in enumerate(zip(res['documents'][0], res['metadatas'][0]))
    ]

In [7]:
import re
from nltk.stem.snowball import SnowballStemmer
import nltk
nltk.download('punkt', quiet=True)

stemmer = SnowballStemmer("spanish")

# Stopwords basicos en español para limpiar ruido en BM25
STOPWORDS_ES = set([
    'de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 
    'para', 'con', 'no', 'una', 'su', 'al', 'es', 'lo', 'como', 'más', 'pero', 'sus', 
    'le', 'ya', 'o', 'este', 'sí', 'porque', 'esta', 'entre', 'cuando', 'muy', 'sin', 
    'sobre', 'también', 'me', 'hasta', 'hay', 'donde', 'quien', 'desde', 'todo', 'nos', 
    'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese', 'eso', 'ante', 
    'ellos', 'e', 'esto', 'mí', 'antes', 'algunos', 'qué', 'unos', 'yo', 'otro', 'otras', 
    'otra', 'él', 'tanto', 'esa', 'estos', 'mucho', 'quienes', 'nada', 'muchos', 'cual', 
    'poco', 'ella', 'estar', 'estas', 'algunas', 'algo', 'nosotros', 'mi', 'mis', 'tú', 
    'te', 'ti', 'tu', 'tus', 'ellas', 'nosotras', 'vosotros', 'vosotras', 'os',
])

def tokenize_bm25(text):
    """Tokeniza texto para BM25: minúsculas, sin puntuación, sin stopwords, con stemming."""
    text = text.lower()
    text = re.sub(r'[^\\w\\s]', '', text)
    words = [w for w in text.split() if w and w not in STOPWORDS_ES]
    return [stemmer.stem(w) for w in words]

# Indice BM25 sobre todos los chunks
all_data  = collection.get()
all_docs  = all_data['documents']
all_metas = all_data['metadatas']

tokenized  = [tokenize_bm25(doc) for doc in all_docs]
bm25_index = BM25Okapi(tokenized)
print(f'Indice BM25: {len(all_docs)} chunks (con stemming)')

def bm25_search(query, k=20):
    scores  = bm25_index.get_scores(tokenize_bm25(query))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {'text': all_docs[i], 'meta': all_metas[i], 'rank_bm25': rank}
        for rank, i in enumerate(top_idx)
    ]

Indice BM25: 268 chunks (con stemming)


In [8]:
def rrf_fusion(dense, bm25, k=60, top_n=5):
    """Reciprocal Rank Fusion."""
    scores = {}

    for item in dense:
        cid = item['meta']['chunk_id']
        scores.setdefault(cid, {'text': item['text'], 'meta': item['meta'], 'score': 0.0})
        scores[cid]['score'] += 1.0 / (k + item['rank_dense'])

    for item in bm25:
        cid = item['meta']['chunk_id']
        scores.setdefault(cid, {'text': item['text'], 'meta': item['meta'], 'score': 0.0})
        scores[cid]['score'] += 1.0 / (k + item['rank_bm25'])

    ranked = sorted(scores.values(), key=lambda x: x['score'], reverse=True)
    return ranked[:top_n]

## 2. Prompt Builder

In [9]:
def build_prompt(query, chunks):
    context_blocks = []
    for i, chunk in enumerate(chunks, 1):
        doc_id = chunk['meta']['doc_id']
        context_blocks.append(f'[Fragmento {i} - Fuente: {doc_id}]\n{chunk["text"]}')

    context = '\n\n'.join(context_blocks)

    return f"""Eres COSORA, un asistente técnico especializado en análisis de actas de obra ferroviaria en España.

REGLAS ESTRICTAS:
1. Responde ÚNICAMENTE con información del CONTEXTO proporcionado.
2. Si la información no está en el contexto, dilo explícitamente.
3. Cita siempre la fuente entre paréntesis: (Fuente: nombre_del_acta).
4. Si varios fragmentos hablan del mismo tema, sintetiza la información en una respuesta coherente, no repitas datos.

FORMATO DE RESPUESTA:
- Usa puntos numerados cuando haya varios aspectos.
- Prioriza: fechas, responsables (DF, UTE, constructora), decisiones tomadas y acciones pendientes.
- Si detectas una evolución temporal (algo cambió entre actas), indícalo cronológicamente.

VOCABULARIO:
- DF = Dirección Facultativa
- UTE = Unión Temporal de Empresas (constructora)
- DEO = Director de Ejecución de Obra
- DO = Director de Obra

=== CONTEXTO ===
{context}

=== PREGUNTA ===
{query}

=== RESPUESTA ==="""

## 3. Generador LLM

In [10]:
if GEN_BACKEND == 'local':
    print(f'Cargando {LOCAL_MODEL}...')
    generator = pipeline(
        'text-generation',
        model=LOCAL_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto',
    )
    print('Modelo cargado')


def generate_answer(prompt):
    if GEN_BACKEND == 'local':
        output = generator(
            prompt,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=generator.tokenizer.eos_token_id,
        )
        return output[0]['generated_text'][len(prompt):].strip()

    elif GEN_BACKEND == 'openai':
        from openai import OpenAI
        api_key = os.getenv('OPENAI_API_KEY')
        if not api_key:
            raise ValueError("No se encontro OPENAI_API_KEY en el .env. Comprueba la ruta en la celda de Setup.")
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model='gpt-4o',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.2,
        )
        return response.choices[0].message.content

    raise ValueError(f'Backend desconocido: {GEN_BACKEND}')

## 4. Pipeline completo

In [11]:
def ask_cosora(query, verbose=True):
    dense  = dense_search(query, k=20)
    bm25   = bm25_search(query, k=20)
    chunks = rrf_fusion(dense, bm25, top_n=10)  # Candidatos iniciales

    # 1. Verificación de Relevancia (Score Thresholding)
    if not chunks or chunks[0]['score'] < 0.015:
        msg = "⚠️ Aviso: No se encontró información suficientemente relevante en las actas para responder a esta pregunta."
        if verbose:
            print(f'Query: {query}')
            print(msg)
            print('-' * 70)
        return msg

    # 2. Filtro dinámico: solo chunks con score >= 60% del mejor
    max_score = chunks[0]['score']
    chunks = [c for c in chunks if c['score'] >= max_score * 0.60]

    # 3. Limpieza del prefijo 'passage: ' para OpenAI
    for c in chunks:
        c['text'] = c['text'].replace('passage: ', '', 1)

    prompt = build_prompt(query, chunks)
    answer = generate_answer(prompt)

    if verbose:
        print(f'Query: {query}')
        print(f'\nChunks recuperados ({len(chunks)}):') 
        for c in chunks:
            print(f"  - [{c['meta']['doc_id']}] score={c['score']:.4f} | {c['text'][:80]}...")
        print(f'\nRespuesta ({GEN_BACKEND}):\n{answer}')
        print('-' * 70)

    return answer

## 5. Tests

In [13]:
queries_test = [
    'Que se decidio sobre el talud?',
    'Cual es el estado del camino provisional?',
    'Que materiales se aprobaron para el drenaje de muros?',
]

for q in queries_test:
    ask_cosora(q)
    print()

Query: Que se decidio sobre el talud?

Chunks recuperados (10):
  - [254275-DO-AVO-15-V07] score=0.0167 | convienen en la necesidad de establecer una reunión de obra con los agentes de S...
  - [244170-DOB-AVO-05-V00-A0] score=0.0167 | |Asistentes | |Objeto | | | |Documentación Entregada | | | | | |Documentación de...
  - [254275-DO-AVO-14-V07] score=0.0164 | que se va a ejecutar está mínimamente en la línea de lo que requiere la estación...
  - [244170-DOB-AVO-05-V00-A0] score=0.0164 | ALCANZADOS / | |COMUNICACIONES E INSTRUCCIONES: | |Revisión temas pendientes en ...
  - [254275-DO-AVO-16-V07] score=0.0161 | UTE que aporte la documentación gráfica que defina el camino provisional ejecuta...
  - [244170-DOB-AVO-07-V00-A0] score=0.0161 | |Asistentes | |Objeto | | | |Documentación Entregada | | | | | |Documentación de...
  - [254275-DO-AVO-15-V07] score=0.0159 | del talud Posterior a la reunión de obra, DF envía en resultado de la verificaci...
  - [244170-DOB-AVO-07-V00-A0] score=0.015

In [12]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

query = "Que se decidio sobre el talud?"
doc_relevante = "3.5. Estabilidad del talud. UTE proporciona un plano..."
doc_irrelevante = "El pavimento definido en la partida de presupuesto..."

for name in ["BSC-LT/MrBERT-es", "intfloat/multilingual-e5-base"]:
    model = SentenceTransformer(name)
    
    # Añadimos los prefijos obligatorios solo para la familia e5
    if "e5" in name:
        q = model.encode(f"query: {query}")
        r = model.encode(f"passage: {doc_relevante}")
        i = model.encode(f"passage: {doc_irrelevante}")
    else:
        q = model.encode(query)
        r = model.encode(doc_relevante)
        i = model.encode(doc_irrelevante)
    
    sim_rel = cosine_similarity([q], [r])[0][0]
    sim_irr = cosine_similarity([q], [i])[0][0]
    print(f"{name}:")
    print(f"  Similaridad con doc relevante:   {sim_rel:.4f}")
    print(f"  Similaridad con doc irrelevante: {sim_irr:.4f}")

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/601M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

...

W0528 17:26:03.795000 2728 torch/_inductor/utils.py:1731] [1/0_1] Not enough SMs to use max_autotune_gemm mode


BSC-LT/MrBERT-es:
  Similaridad con doc relevante:   0.9160
  Similaridad con doc irrelevante: 0.9204


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


intfloat/multilingual-e5-base:
  Similaridad con doc relevante:   0.8128
  Similaridad con doc irrelevante: 0.7689
